# Text als Daten

Dieser Kurs verwendet das Nachrichtenarchiv des Projekts orf-teletext - Schlagzeilen und Teaser von ORF, Der Standard, tagesschau und Handelsblatt -, um das Standardwerkzeug des Text Mining durchzuarbeiten, vom Zählen von Wörtern bis zum Trainieren von Klassifikatoren. Sechs Skripte lang bleiben wir bewusst bei **klassischen, nachvollziehbaren Methoden**; ein großes Sprachmodell (LLM) kommt erst im letzten Skript vor, wo es mit allem verglichen werden kann, was wir davor aufgebaut haben.

Am Ende dieses Skripts kannst du:

- entscheiden, was als **Dokument** zählt, und verstehen, warum das die erste und folgenreichste Entscheidung jeder Textanalyse ist;
- deutschen Text mit einem regulären Ausdruck **tokenisieren** und erklären, wo naive Ansätze scheitern;
- **Normalisierung** und **Stoppwort-Entfernung** anwenden und wissen, was dabei jeweils verloren geht;
- ein Korpus mit **Häufigkeitsverteilungen**, dem Zipfschen Gesetz und N-Grammen beschreiben.

Alles in diesem Skript verwendet nur die Standardbibliothek, pandas und matplotlib (dazu die Stoppwortliste von NLTK). Ein Sprachmodell kommt noch nicht vor - das folgt in Skript 02 -, daher ist jeder Schritt hier sichtbar, statt in einem Bibliotheksaufruf zu verschwinden.

## Voraussetzungen

In [ ]:
pip list

Das Korpus stammt aus dem Nachrichtenarchiv des Projekts orf-teletext. Das Hilfsmodul `textmining.py` liest es aus `D:\projects\orf-teletext` oder, falls gesetzt, aus dem Ordner in der Umgebungsvariable `TEXTMINING_DATA`.

## Präambel

Wir laden alle Pakete und Module, die wir im Folgenden benötigen.

In [ ]:
# Reguläre Ausdrücke und Zähler aus der Standardbibliothek
import re
from collections import Counter

# Visualisierung und Datentransformation
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.dataset as ds

# Paket für die Stoppwortliste
import nltk

# Hilfsmodul des Kurses: Laden und Bereinigen des Korpus
import textmining as tm

# Darstellungsoptionen für Tabellen und Grafiken
pd.set_option("display.max_colwidth", 100)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## Von Abrufen zu Dokumenten

Das Archiv speichert alle fünfzehn Minuten, wie die Feeds aussahen. Eine Schlagzeile, die einen Tag lang online war, wurde fast hundertmal abgerufen. Diese Wiederholung ist wertvoll, um zu untersuchen, *wie sich Feeds verhalten*, für Text Mining ist sie aber Gift: Jedes Wort einer langlebigen Schlagzeile würde einmal pro Abruf gezählt, und das Korpus würde vor allem messen, wie lange Meldungen online blieben.

Die **Silver**-Tabelle des Projekts hat die Abrufe bereits auf eine Zeile pro Eintrag reduziert. Vergleichen wir die beiden:

In [ ]:
# Zähle die Zeilen in Bronze (Abrufe) und Silver (Einträge)
bronze_rows = ds.dataset(tm.ROOT / "bronze", format="parquet").count_rows()
silver = pd.read_parquet(tm.SILVER)
print(f"bronze: {bronze_rows:>9,} Zeilen  (eine pro Eintrag und Abruf)")
print(f"silver: {len(silver):>9,} Zeilen  (eine pro Eintrag)")
print(f"jeder Eintrag kommt im Schnitt in {bronze_rows / len(silver):.0f} Abrufen vor")

Silver ist der Ausgangspunkt, aber noch keine saubere Menge von Dokumenten. Es braucht drei weitere Entscheidungen, und `textmining.load_corpus()` - die gemeinsame Ladefunktion, die jedes Skript verwendet - trifft alle drei. Hier betrachten wir sie einzeln, damit sichtbar wird, was jede davon entfernt.

### Entscheidung 1: Eine URL ist nicht immer eine Meldung

Manche Medien verwenden für ein wiederkehrendes Format immer dieselbe Kennung. Das tägliche Markt-Briefing des Handelsblatts behält seine URL und bekommt jeden Morgen ein neues Datum im Titel. Für Silver sieht das aus wie ein Eintrag, der wochenlang täglich überarbeitet wurde; für Text Mining sind es mehrere voneinander unabhängige Texte, die aneinandergeklebt wurden. Silver markiert diese Einträge: Sie verschwanden aus dem Feed und kamen zurück (`contiguous` ist false), mit einem neuen Zeitstempel (`republished` ist true).

In [ ]:
# Einträge, die den Feed verlassen haben und neu veröffentlicht wurden
composite = ~silver["contiguous"] & silver["republished"]
silver.loc[composite, ["feed", "first_title", "final_title", "published_versions"]]

### Entscheidung 2: Derselbe Artikel unter mehreren URLs

Auch das umgekehrte Problem gibt es. Das Handelsblatt veröffentlicht manche Artikel unter einer neuen URL erneut, sodass dieselbe Schlagzeile als mehrere Einträge auftaucht. Würden wir jede Kopie zählen, bekämen die Wörter dieses Artikels das doppelte oder dreifache Gewicht aller anderen.

In [ ]:
# Finde Schlagzeilen, die im selben Feed mehrfach vorkommen
dupes = silver[silver.duplicated(["feed", "final_title"], keep=False)]
dupes.groupby(["feed", "final_title"]).size().sort_values(ascending=False).head(8)

### Entscheidung 3: Markup ist kein Text

Feed-Beschreibungen sind für die Anzeige im Browser gedacht, und manche enthalten HTML. Hier eine unbearbeitete Beschreibung von derStandard:

In [ ]:
# Zeige eine Beschreibung vor und nach der HTML-Bereinigung
raw = silver.loc[silver["feed"] == "derstandard_wirtschaft", "final_description"].iloc[0]
print(raw)
print()
print("bereinigt:", tm.clean_html(raw))

Ohne Bereinigung würde ein Tokenisierer diese Bild-URL in Wörter wie `lido`, `images`, `jpeg` und `rs` zerlegen - die in *jedem* derStandard-Eintrag vorkommen, sodass jeder spätere Vergleich der Medien "entdecken" würde, dass derStandard von JPEGs handelt.

### Und ein Ausschluss: die Sprache

Der Feed der EZB ist auf Englisch. Jedes auf Deutsch zugeschnittene Werkzeug in diesem Kurs - die Stoppwortliste, das spaCy-Modell, die Behandlung von Komposita - würde ihn falsch verarbeiten, daher behält die Ladefunktion nur deutschsprachige Feeds, sofern nicht anders angegeben.

### Das Ergebnis

In [ ]:
# Lade das bereinigte Korpus und zeige einen Überblick pro Medium
corpus = tm.load_corpus()
print(f"{len(corpus):,} Dokumente")
corpus.groupby("outlet").agg(
    Land=("country", "first"),
    Meldungen=("title", "size"),
    mit_Beschreibung=("description", lambda s: (s != "").mean()),
).style.format({"mit_Beschreibung": "{:.0%}"})

Der ORF fällt auf: Die meisten seiner Einträge sind kurze *Ticker*-Schlagzeilen ohne Teaser-Text. Diese Asymmetrie spielt im ganzen Kurs eine Rolle. Wann immer wir Medien anhand von **Titel plus Beschreibung** (der Spalte `text`) vergleichen, trägt der ORF pro Eintrag weit weniger Text bei als die anderen. Vergleiche anhand der **Titel allein** sind fairer, und diese verwenden die meisten Skripte.

In [ ]:
# Median der Wortanzahl in Titel und Beschreibung pro Medium
words = corpus.assign(
    title_words=corpus["title"].str.split().str.len(),
    description_words=corpus["description"].str.split().str.len(),
)
words.groupby("outlet")[["title_words", "description_words"]].median()

## Tokenisierung

Die **Tokenisierung** zerlegt einen String in die Einheiten, die wir zählen wollen. Das klingt trivial, ist aber die Stelle, an der der meiste unbemerkte Schaden entsteht. Der naheliegende Ansatz trennt an Leerzeichen:

In [ ]:
# Wähle eine Schlagzeile mit typografischen Anführungszeichen und Bindestrich
example = corpus.loc[corpus["title"].str.contains("„") & corpus["title"].str.contains("-"), "title"].iloc[0]
print(example)
print(example.split())

Satzzeichen bleiben an ihren Nachbarn hängen: Ein Wort mit nachfolgendem Doppelpunkt und dasselbe Wort am Satzende werden zu verschiedenen Tokens, und deutsche typografische Anführungszeichen („ “) kleben an den Wörtern, die sie umschließen. Die Zählungen zerfallen, ohne dass ein Fehler gemeldet wird.

Ein **Tokenisierer auf Basis eines regulären Ausdrucks** beschreibt, was ein Wort *ist*, statt was Wörter trennt. Das naheliegende englische Muster ist eine Falle:

In [ ]:
naive = re.compile(r"[A-Za-z]+")
print(naive.findall("Für Österreichs Bürger: Spritpreise über 2 Euro"))

Aus `Für` wurde `F` + `r`, `Österreichs` verlor seinen ersten Buchstaben, und es wurde kein Fehler gemeldet. Umlaute und ß sind im Deutschen gewöhnliche Buchstaben und müssen in der Zeichenklasse enthalten sein. Pythons `\w` deckt sie bereits ab (es ist Unicode-fähig), passt aber auch auf Ziffern und Unterstriche, daher schreiben wir die Klasse als `[^\W\d_]` aus: "Wortzeichen ohne Ziffern und Unterstrich".

Zwei weitere Entscheidungen, die das Deutsche betreffen:

- **Komposita mit Bindestrich bleiben ganz.** `Sachsen-Anhalt`, `CDU-Chef` und `Nahost-Konflikt` sind jeweils ein einziges Konzept. Eine Trennung am Bindestrich würde `Sachsen` und `Anhalt` getrennt zählen und zwei unzusammenhängende Begriffe erfinden.
- **Zahlen fallen vorerst weg.** Preise, Datumsangaben und Indexstände (`25.000`, `5,3`) sind bedeutsam, als Vokabular erzeugen sie aber vor allem Rauschen. Skript 02 zeigt, wie ein Sprachmodell sie als Zahlen markieren kann, statt sie blind zu verwerfen.

In [ ]:
# Ein Token: Buchstabenfolge, optional mit Bindestrich verbundene weitere Buchstabenfolgen
TOKEN_RE = re.compile(r"[^\W\d_]+(?:-[^\W\d_]+)*")


def tokenize(text: str) -> list[str]:
    return TOKEN_RE.findall(text)


print(tokenize("Für Österreichs Bürger: Spritpreise über 2 Euro"))
print(tokenize(example))

## Normalisierung

Die **Normalisierung** legt fest, welche Oberflächenformen als dasselbe Wort zählen sollen. Der übliche erste Schritt, die Kleinschreibung, scheint im Deutschen teuer, weil **jedes Nomen großgeschrieben wird**: Nur die Groß- und Kleinschreibung unterscheidet *Laut* (ein Geräusch) von *laut* (nicht leise, oder "gemäß"), oder *Essen* (die Mahlzeit) von *essen* (das Verb).

Bevor wir entscheiden, sehen wir uns an, was in diesem Korpus tatsächlich zusammenfällt. Die folgende Zelle findet jedes Wort, das sowohl großgeschrieben als auch kleingeschrieben vorkommt, und ordnet die Paare danach, wie oft die seltenere Form vorkommt.

In [ ]:
# Zähle alle Tokens in den Titeln
title_tokens = Counter(t for title in corpus["title"] for t in tokenize(title))

# Finde Wörter, die groß- und kleingeschrieben vorkommen
collisions = pd.DataFrame(
    [
        (word, n, word.lower(), title_tokens[word.lower()])
        for word, n in title_tokens.items()
        if word[0].isupper() and word.lower() != word and title_tokens[word.lower()]
    ],
    columns=["capitalised", "n_cap", "lowercase", "n_lower"],
)

# Ordne nach der Häufigkeit der selteneren Form
collisions.assign(rarer=collisions[["n_cap", "n_lower"]].min(axis=1)).nlargest(12, "rarer")

Die Kollisionen sind ganz überwiegend **Großschreibung am Satzanfang**, keine Paare aus Nomen und Verb: *Die* und *die*, *Nach* und *nach*, *Neue* und *neue* sind dasselbe Wort, großgeschrieben, weil es eine Schlagzeile eröffnet. Würden wir die Schreibung beibehalten, würde jedes davon in zwei Wörter zerfallen. Echte bedeutungstragende Paare gibt es, sie sind im Vergleich aber selten.

Für das Zählen von **Themen** in Schlagzeilen ist Casefolding (Kleinschreibung) also die richtige Wahl, und so gehen wir vor. Wo die Wortart wirklich wichtig ist, bietet Skript 02 ein besseres Werkzeug als die Schreibung: einen Wortart-Tagger (POS-Tagger), der das Nomen *Laut* anhand des Kontexts von der Präposition *laut* unterscheiden kann.

Pythons `str.casefold()` ist dabei die richtige Wahl statt `lower()`: Es ist die gründlichere, Unicode-fähige Variante, die für den Vergleich ohne Berücksichtigung der Schreibung gedacht ist. Im Deutschen bildet sie außerdem `ß` auf `ss` ab, sodass `Straße` und die Schweizer Schreibweise `Strasse` zusammenfallen.

In [ ]:
# Vergleiche lower() und casefold()
print("Straße".lower(), "Straße".casefold(), "STRASSE".casefold())

## Stoppwörter

**Stoppwörter** sind häufige Wörter mit wenig thematischem Gehalt: `der`, `und`, `mit`. Wenn wir sie entfernen, sehen nicht mehr alle Häufigkeitstabellen gleich aus. NLTK liefert eine deutsche Liste mit:

In [ ]:
# Lade die deutsche Stoppwortliste (und lade sie bei Bedarf herunter)
try:
    from nltk.corpus import stopwords

    GERMAN_STOPWORDS = set(stopwords.words("german"))
except LookupError:
    nltk.download("stopwords", quiet=True)
    from nltk.corpus import stopwords

    GERMAN_STOPWORDS = set(stopwords.words("german"))

print(len(GERMAN_STOPWORDS), "Stoppwörter, z. B.", sorted(GERMAN_STOPWORDS)[:12])
print("Negationen in der Liste:", sorted(GERMAN_STOPWORDS & {"nicht", "kein", "keine", "nichts", "nie"}))

Sieh dir immer an, was eine Stoppwortliste löscht. Diese hier entfernt Negationen, was für das Zählen von Themen harmlos ist und für alles, was mit Bedeutung oder Haltung zu tun hat, verheerend: *"keine Einigung"* und *"Einigung"* würden identisch.

Fügen wir die Schritte zusammen, erhalten wir die Vorverarbeitungsfunktion, die wir im Rest dieses Skripts verwenden:

In [ ]:
# Minimale Tokenlänge
MIN_LENGTH = 2


# Tokenisiere, wende Casefolding an und entferne Stoppwörter und zu kurze Tokens
def prepare(text: str) -> list[str]:
    tokens = (t.casefold() for t in tokenize(text))
    return [t for t in tokens if t not in GERMAN_STOPWORDS and len(t) >= MIN_LENGTH]


corpus["tokens"] = corpus["title"].map(prepare)
corpus[["outlet", "title", "tokens"]].sample(5, random_state=3)

## Häufigkeitsverteilungen

Mit einem sauberen Token-Strom braucht das Zählen nicht mehr als `collections.Counter`.

In [ ]:
# Die n häufigsten Terme eines Teilkorpus
def top_terms(frame: pd.DataFrame, n: int = 12) -> pd.Series:
    counts = Counter(t for tokens in frame["tokens"] for t in tokens)
    return pd.Series(dict(counts.most_common(n)))


# Eine Spalte pro Medium
pd.DataFrame(
    {outlet: top_terms(group).index for outlet, group in corpus.groupby("outlet")}
)

Schon diese grobe Tabelle trennt die Medien. Die beiden allgemeinen Nachrichten-Feeds (ORF, tagesschau) werden von Politik und Konflikten dominiert - Staatschefs, Länder, Opfer -, während die beiden Wirtschaftsredaktionen Geld und Märkte in den Vordergrund rücken. Auch das Heimatland jedes Mediums scheint durch (`österreich`, `wien` gegenüber `deutschland`, `berlin`). Die Tabelle zeigt aber auch die Grenzen roher Zählungen: Die Terme, die *überall* häufig sind, verdrängen jene, die jedes Medium unverwechselbar machen. Skript 03 geht das mit TF-IDF und Keyness-Statistiken an.

### Zipfsches Gesetz

Worthäufigkeiten folgen in jeder natürlichen Sprache einem auffallend regelmäßigen Muster: Das *r*-häufigste Wort kommt ungefähr proportional zu *1/r* vor. In einer doppelt logarithmischen Darstellung der Häufigkeit gegen den Rang ergibt das eine Gerade mit einer Steigung nahe -1.

In [ ]:
# Häufigkeiten aller Terme, absteigend sortiert
counts = Counter(t for tokens in corpus["tokens"] for t in tokens)  # wird im Abschnitt "Das Problem der Komposita" wiederverwendet
freqs = np.array(sorted(counts.values(), reverse=True))
ranks = np.arange(1, len(freqs) + 1)

# Passe eine Gerade im doppelt logarithmischen Raum an
slope, intercept = np.polyfit(np.log(ranks), np.log(freqs), 1)

# Zeichne Häufigkeit gegen Rang
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.loglog(ranks, freqs, ".", ms=3, color="#2F6690", alpha=0.6, label="beobachtet")
ax.loglog(ranks, np.exp(intercept) * ranks**slope, color="#B33A3A", label=f"Anpassung, Steigung {slope:.2f}")
ax.set(xlabel="Rang", ylabel="Häufigkeit", title="Zipfsches Gesetz im Schlagzeilen-Korpus")
ax.legend()
plt.show()

# Zähle die Terme, die genau einmal vorkommen (Hapax legomena)
hapax = sum(1 for c in counts.values() if c == 1)
print(f"{len(counts):,} verschiedene Terme, davon {hapax:,} ({hapax / len(counts):.0%}) genau einmal")

Daraus folgen zwei praktische Konsequenzen, die jedes weitere Skript prägen:

- **Der Großteil des Vokabulars ist selten.** In einem Korpus dieser Größe sind weit mehr als die Hälfte aller verschiedenen Terme *Hapax legomena* - Wörter, die genau einmal vorkommen. Sie liefern wenig statistische Evidenz, weshalb spätere Skripte oft Terme unter einer minimalen Dokumenthäufigkeit verwerfen.
- **Wenige Wörter leisten den Großteil der Arbeit.** Der Kopf der Verteilung ist eine kurze Liste sehr häufiger Terme, weshalb die Entfernung von Stoppwörtern die Zählungen so stark verändert, obwohl dabei so wenige verschiedene Wörter wegfallen.

Auch die Abflachung ganz oben ist üblich: Die Entfernung der Stoppwörter hat den Kopf abgeschnitten, also genau den Bereich, in dem der steilste Teil der Kurve läge.

## N-Gramme

Ein **N-Gramm** ist eine Folge von *n* benachbarten Tokens. Bigramme stellen Namen und feste Wendungen wieder her, die einzelne Tokens auseinanderreißen.

In [ ]:
# Zähle Paare benachbarter Tokens
def bigram_counts(frame: pd.DataFrame) -> Counter:
    return Counter(
        f"{a} {b}" for tokens in frame["tokens"] for a, b in zip(tokens, tokens[1:])
    )


# Die zehn häufigsten Bigramme pro Medium, mit Häufigkeit in Klammern
pd.DataFrame(
    {
        outlet: [f"{bg} ({n})" for bg, n in bigram_counts(group).most_common(10)]
        for outlet, group in corpus.groupby("outlet")
    }
)

Die Bigramme werden hier aus der *gefilterten* Tokenliste gebildet, sodass zwei Wörter, die durch ein Stoppwort getrennt waren, nebeneinander rücken. Für thematische Wendungen ist das oft erwünscht - `strasse hormus` ist die *Straße von Hormus* ohne das `von` und als eine Wendung besser aufgehoben -, es kann aber auch Paare erzeugen, die im Original nie nebeneinander standen. Das solltest du beim Lesen der Tabelle oben im Hinterkopf behalten.

### Eine Konvention deutscher Schlagzeilen: die Dachzeile (Kicker)

Deutsche Schlagzeilen beginnen oft mit einem kurzen Themenetikett, gefolgt von einem Doppelpunkt - der *Dachzeile* oder dem Kicker: *"Immobilien: …"*, *"Nahost: …"*. Wie oft ein Medium das tut, ist ein stilistischer Fingerabdruck, und es wirkt sich direkt auf das Text Mining aus, denn das Wort in der Dachzeile ist faktisch ein Kategorie-Label mitten im Text.

In [ ]:
# Extrahiere die Dachzeile: alles vor dem ersten Doppelpunkt am Titelanfang
kicker = corpus["title"].str.extract(r"^([^:]{2,30}):\s", expand=False)
corpus["kicker"] = kicker

# Anteil der Titel mit Dachzeile und die häufigsten Dachzeilen pro Medium
summary = corpus.groupby("outlet")["kicker"].agg(Anteil=lambda s: s.notna().mean())
summary["häufigste Dachzeilen"] = corpus.groupby("outlet")["kicker"].agg(
    lambda s: ", ".join(s.value_counts().head(5).index)
)
summary.style.format({"Anteil": "{:.0%}"})

Vor allem das Handelsblatt stellt fast jeder Schlagzeile seinen Ressortnamen voran. Das solltest du dir merken: Jede Methode, die Medien vergleicht, wird diese Etiketten aufgreifen, und ein Klassifikator, der auf Handelsblatt-Schlagzeilen trainiert wird, würde die Antwort teilweise am ersten Wort ablesen.

## Das Problem der Komposita

Das Deutsche schreibt als ein Wort, was das Englische als Wortgruppe schreibt. Hier sind die längsten Tokens im Korpus:

In [ ]:
# Zeige die zehn längsten Tokens mit ihrer Länge
longest = sorted(counts, key=len, reverse=True)[:10]
for token in longest:
    print(f"{len(token):3}  {token}")

Jedes davon besteht aus mehreren zusammengeschweißten Konzepten. Für einen Zähler sind `spritpreise` und `spritpreisbremse` unzusammenhängende Strings. Ein Thema, das in mehreren verschiedenen Komposita ausgedrückt wird, verteilt sich auf Terme, die jeweils zu selten sind, um aufzufallen.

Die einfachste Abhilfe ist, nach einem Bruchstück *innerhalb* der Tokens zu suchen:

In [ ]:
# Alle Tokens, die ein Bruchstück enthalten, nach Häufigkeit sortiert
def compounds_containing(fragment: str) -> pd.Series:
    return pd.Series({t: c for t, c in counts.items() if fragment in t}).sort_values(ascending=False)


sprit = compounds_containing("sprit")
print(f"exaktes Token 'spritpreise': {counts['spritpreise']}  |  alle Tokens, die 'sprit' enthalten: {sprit.sum()}")
sprit.head(12)

Die Suche nach Teilstrings ist grob - achte in der Liste auf `giftspritzen`, das nur deshalb gefunden wird, weil *Spritze* die Zeichenfolge *sprit* enthält -, sie zeigt aber, wie stark ein Thema bei exakter Übereinstimmung unterzählt wird. Echte Kompositumzerlegung (`Spritpreisbremse` in `Sprit`, `Preis`, `Bremse` aufzuteilen) braucht ein Wörterbuch oder ein statistisches Zerlegungsverfahren. Skript 05 umgeht das Problem mit **Zeichen-N-Grammen**, die in Wörter hineinsehen, ohne sie zu zerlegen.

## Zusammenfassung

| Schritt | Was er behebt | Was er kostet |
|---|---|---|
| eine Zeile pro Eintrag, dedupliziert | wiederholte Abrufe und Kopien, die Zählungen aufblähen | nichts, wenn richtig gemacht |
| HTML-Bereinigung | Markup, das sich als Vokabular ausgibt | nichts |
| Tokenisierung mit regulärem Ausdruck | Satzzeichen, die Wörter zersplittern | Zahlen, sofern nicht gesondert behandelt |
| Casefolding | Großschreibung am Satzanfang, die Wörter aufspaltet | Unterscheidung von Nomen und Verb |
| Stoppwort-Entfernung | Funktionswörter, die die Zählungen dominieren | Negation, für alles, was mit Bedeutung zu tun hat |

Jeder dieser Schritte ist eine **Entscheidung**, keine Voreinstellung. Das nächste Skript ersetzt mehrere davon durch ein Sprachmodell, das die deutsche Grammatik kennt, und zeigt, welche der obigen Probleme es tatsächlich löst.

## Übungen

1. **Vokabular der Beschreibungen.** Baue `tokens` aus der Spalte `text` statt aus `title` neu auf und erstelle die Tabelle der häufigsten Terme erneut. Welches Medium verändert sich am stärksten, und warum würdest du das nach dem Korpus-Überblick im Abschnitt "Von Abrufen zu Dokumenten" erwarten?
2. **Eine eigene Stoppwortliste.** Die NLTK-Liste wurde nicht für Schlagzeilen gemacht. Finde die zehn häufigsten verbleibenden Tokens über alle Medien hinweg und entscheide für jedes, ob es ein Themenwort oder Schlagzeilen-Füllmaterial ist. Was würdest du ergänzen?
3. **Dachzeilen als Labels.** Entferne die Dachzeile (alles bis zum ersten Doppelpunkt) aus den Handelsblatt-Titeln vor der Tokenisierung. Wie stark verändert sich die Liste der häufigsten Terme?
4. **Heapssches Gesetz.** Die Größe des Vokabulars wächst mit der Größe des Korpus, aber immer langsamer. Zeichne die Anzahl verschiedener Tokens gegen die Anzahl bisher gesehener Tokens, wobei du die Schlagzeilen in der Reihenfolge ihrer Veröffentlichung hinzufügst. Flacht die Kurve ab?

## Quellen

* https://www.nltk.org/
* https://pandas.pydata.org/docs/
* https://matplotlib.org/
* https://de.wikipedia.org/wiki/Zipfsches_Gesetz